# The ReAct agent

Think, call a tool, read, repeat; then check the answer against what was retrieved.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

## The loop with a scripted model (offline)
The tests drive the real loop and real tools with a model that answers from a script.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_mcp_adapters.tools import load_mcp_tools
from fastmcp import Client
from marginalia.agent.react_agent import BookAgent
from marginalia.mcp_server.server import build_server
from tests.conftest import book_retriever, synthetic_profile
from tests.test_agent import ScriptedModel, script, tool_call

server = build_server(synthetic_profile(), lambda: book_retriever('agent-notebook'))
# The agent first searches with the question itself; the script then plays the model's own turns.
steps = script(None, tool_call('search_passages', query='Lazarus Gospel', part=4, chapter=4),
               AIMessage('Sonia read the raising of Lazarus [Part IV, Chapter IV].'))
async with Client(server) as c:
    agent = BookAgent(ScriptedModel(respond=steps, calls=[]), await load_mcp_tools(c.session), profile=synthetic_profile())
    result = await agent.ask('What did Sonia read?', on_message=lambda m: print(type(m).__name__, str(m.content)[:80] or m.tool_calls))
print('\nanswer:', result.answer)
print('tool calls:', result.tool_calls)
print('citations checked:', result.citations, '| unknown:', result.unknown_citations)

## The system prompt the model gets

In [ ]:
print(agent.system_prompt)

## The real agent (needs Ollama and an indexed book)
qwen3:4b thinks before it answers; the first search uses the question itself; the grader checks the answer; tokens are counted per role.

In [ ]:
from marginalia.agent.react_agent import book_agent
from marginalia.agent.sampling import Sampling
from marginalia.library import Library
books = [b for b in Library().books() if b.status == 'ready']
if books:
    async with book_agent(books[0], max_tool_calls=4) as real_agent:
        res = await real_agent.ask('What happens in the first chapter?')
    print(res.answer, '\n', res.citations, '\nsupported:', res.supported, '\ntokens:', res.usage)
else:
    print('no ready book')

## Sampling
Readers may change these per question; the grader keeps the server's settings.

In [ ]:
from marginalia.config import Settings
print(Sampling(temperature=0.9, max_searches=3).resolve(Settings()))